# Engineer causal aircraft-rotation features

Extend the shared departure feature table along a separate, append-only experiment path. For each target flight, the scheduled-departure timestamp is the prediction cutoff. The aircraft tail number is used to find the immediately preceding scheduled JFK event. A preceding arrival is a rotation match; a preceding JFK departure blocks older, stale arrivals.

The inbound leg's realized arrival delay and actual turn time are exposed only when that aircraft had arrived by the target cutoff. If it had not arrived, the dataset records only the observable not-arrived state and scheduled overdue duration. Learned preprocessing remains in the later model pipeline.

In [ ]:
YEAR = 2019

AIRPORT = "JFK"

## Configure the departure and inbound sources

The target rows come from the unchanged shared departure feature file. The inbound rotation history comes from the concatenated cleaned BTS table for flights arriving at the configured airport. The output is separately named so existing feature datasets and experiment results remain intact.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "notebooks" and (Path.cwd().parent / "data").is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from feature_engineering import MODEL_TARGETS
from feature_engineering_rotation import (
    ROTATION_AUDIT_COLUMNS,
    ROTATION_FEATURES,
    ROTATION_OUTPUT_COLUMNS,
    add_departure_rotation_features,
    validate_departure_rotation_features,
)

AIRPORT = str(AIRPORT).strip().upper()
YEAR = int(YEAR)
DEPARTURE_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures.csv"
INBOUND_FILE = PROJECT_ROOT / f"data/bts/cleaned_{AIRPORT}_{YEAR}.csv"
OUTPUT_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures_rotation.csv"

print(pd.Series({
    "departures": str(DEPARTURE_FILE),
    "inbound history": str(INBOUND_FILE),
    "output": str(OUTPUT_FILE),
}))

## Load and validate both flight populations

The notebook validates rather than filters either source. The departure file must contain only flights leaving the configured airport and the inbound table only flights arriving there. Both must cover the requested year.

In [ ]:
for source_file in [DEPARTURE_FILE, INBOUND_FILE]:
    if not source_file.is_file():
        raise FileNotFoundError(f"Required rotation source does not exist: {source_file}")

departures = pd.read_csv(DEPARTURE_FILE, low_memory=False)
inbound = pd.read_csv(INBOUND_FILE, low_memory=False)

departure_required = {"Year", "Origin", "DATE", "Tail_Number", MODEL_TARGETS["1A"]}
inbound_required = {
    "Year", "FlightDate", "Reporting_Airline", "Tail_Number",
    "Flight_Number_Reporting_Airline", "Origin", "Dest", "DATE",
    "CRSArrTime", "CRSElapsedTime", "ArrDelay", "ArrDel15",
}
if departure_required - set(departures.columns):
    raise KeyError(f"Departure feature data is missing: {sorted(departure_required - set(departures.columns))}")
if inbound_required - set(inbound.columns):
    raise KeyError(f"Inbound BTS data is missing: {sorted(inbound_required - set(inbound.columns))}")
if set(ROTATION_OUTPUT_COLUMNS) & set(departures.columns):
    raise ValueError("Input already contains rotation fields; use the base departure feature file")

departure_origin = departures["Origin"].astype("string").str.strip().str.upper()
inbound_destination = inbound["Dest"].astype("string").str.strip().str.upper()
departure_year = pd.to_numeric(departures["Year"], errors="coerce")
inbound_year = pd.to_numeric(inbound["Year"], errors="coerce")
if not departure_origin.eq(AIRPORT).all():
    raise ValueError(f"Departure input contains origins other than {AIRPORT}")
if not inbound_destination.eq(AIRPORT).all():
    raise ValueError(f"Inbound input contains destinations other than {AIRPORT}")
if not departure_year.eq(YEAR).all() or not inbound_year.eq(YEAR).all():
    raise ValueError(f"A rotation input contains years other than {YEAR}")

target = pd.to_numeric(departures[MODEL_TARGETS["1A"]], errors="coerce")
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError("DepDel15 must be complete and binary")

print(pd.Series({
    "departure rows": len(departures),
    "departure columns": len(departures.columns),
    "inbound rows": len(inbound),
    "inbound columns": len(inbound.columns),
    "delayed departures": int(target.sum()),
}))

## Add rotation features

Airport-local schedule clocks are converted to UTC before leg ordering. Scheduled inbound arrival dates are reconstructed by choosing the destination-local date whose UTC elapsed time best agrees with BTS `CRSElapsedTime`. The reconstruction residual remains in the output for audit.

In [ ]:
features = add_departure_rotation_features(
    departures, inbound, airport=AIRPORT
)
if len(features) != len(departures):
    raise ValueError("Rotation feature engineering changed the departure row count")
if not features[departures.columns].equals(departures):
    raise ValueError("Rotation feature engineering changed one or more source columns")
if list(features.columns[-len(ROTATION_OUTPUT_COLUMNS):]) != ROTATION_OUTPUT_COLUMNS:
    raise ValueError("Rotation output columns were not appended in the documented order")

feature_validation = validate_departure_rotation_features(features)
feature_validation

## Review match coverage and information content

The target-rate summary is descriptive only; it does not select features or tune a model. A large rate for `NOT_ARRIVED` is plausible because that state directly means the assigned aircraft is not at the gate by scheduled departure. This information is causal at the cutoff but requires a timely aircraft-assignment and arrival-event feed in deployment.

In [ ]:
status_summary = (
    features.assign(_TARGET=target)
    .groupby("ROTATION_STATUS", dropna=False, observed=True)
    .agg(rows=("_TARGET", "size"), delayed=("_TARGET", "sum"), delay_rate=("_TARGET", "mean"))
    .sort_values("rows", ascending=False)
)
status_summary["row_percent"] = status_summary["rows"] / len(features) * 100
status_summary

In [ ]:
timing_audit = pd.Series({
    "rotation matches": int(features["ROTATION_MATCH_FOUND"].sum()),
    "arrived by cutoff": int(features["ROTATION_INBOUND_ARRIVED_BY_CUTOFF"].sum()),
    "not arrived by cutoff": int(features["ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF"].sum()),
    "actual outcomes visible before cutoff violations": int(
        features.loc[
            features["ROTATION_INBOUND_ARRIVED_BY_CUTOFF"].eq(0),
            ["ROTATION_INBOUND_ARR_DELAY", "ROTATION_INBOUND_DELAYED_15",
             "ROTATION_ACTUAL_TURN_MINUTES", "ROTATION_PRIOR_ACTUAL_ARRIVAL_UTC"],
        ].notna().any(axis=1).sum()
    ),
    "schedule reconstructions with residual over 1 minute": int(
        pd.to_numeric(
            features["ROTATION_SCHEDULE_RECONSTRUCTION_ERROR_MINUTES"],
            errors="coerce",
        ).gt(1).sum()
    ),
}, name="rotation timing validation")
timing_audit

## Save the rotation-enhanced departure dataset

Retain every source and audit column, append the rotation fields, and write a separate annual dataset. The standard departure and backlog files remain separate experiment paths.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT_FILE, index=False)

summary = pd.Series({
    "airport": AIRPORT,
    "year": YEAR,
    "rows": len(features),
    "columns": len(features.columns),
    "rotation features": len(ROTATION_FEATURES),
    "rotation audit columns": len(ROTATION_AUDIT_COLUMNS),
    "target": MODEL_TARGETS["1A"],
    "output": str(OUTPUT_FILE),
}, name="departure rotation feature summary")
print(f"Saved {len(features):,} rows to {OUTPUT_FILE}")
summary